# Báo Cáo Chi Tiết Quá Trình Xây Dựng & Tối Ưu Hệ Thống Phát Hiện Té Ngã (`fall_pose_v2`)

**Đề tài:** Hệ thống nhận diện hành vi và phát hiện té ngã thời gian thực kết hợp cảnh báo IoT (ESP32)  
**Kiến trúc:** Two-Stage Temporal Pose Pipeline (`YOLOv8n-Pose` $\rightarrow$ `66-Dim Geometric & Kinematic Features` $\rightarrow$ `FallNet BiLSTM / Light Transformer` $\rightarrow$ `AlertStateMachine`)

---

## 1. Đặt Vấn Đề & Phân Tích Nguyên Nhân Lỗi Ở Phiên Bản Cũ (`v1`)

Trong quá trình phát triển đồ án, hệ thống đã trải qua **3 giai đoạn tiến hóa chính**:

1. **Hướng tiếp cận ban đầu (YOLO thuần trên từng ảnh tĩnh & luật khớp xương thủ công):**
   - Chỉ nhìn từng khung hình đơn lẻ (Single-frame bounding box) hoặc đặt ngưỡng góc từng khớp xương bằng `if/else`.
   - **Nhược điểm:** Rất dễ sai lệch khi thay đổi góc camera, khoảng cách đứng hoặc khi người bị che khuất một phần cơ thể.

2. **Phiên bản `v1` (`fall_pose_pipeline` — 3 lớp `normal / warning / fall`):**
   - Khi kiểm thử thực tế với webcam, phiên bản `v1` gặp **2 lỗi nghiêm trọng**:
     1. **Báo động giả khi người chỉ đi ngang qua camera:** Trong file suy luận cũ có gán cứng luật ghi đè (`if tilt_deg >= 30.0 or aspect_ratio >= 0.90: warning`). Khi người đi ngang qua gần webcam, sải tay hoặc khung hình bị cắt nửa người làm tỷ lệ `width/height >= 0.90`, gây báo động giả ngay lập tức.
     2. **Nhầm lẫn giữa Nằm (`lie`) và Ngã (`fall`) + Rò rỉ dữ liệu (`Data Leakage`):**
        - Trong bộ URFD gốc, toàn bộ các frame sau khi ngã (khi người đã nằm yên trên sàn hàng chục giây) đều bị gán nhãn `1 (fall)`. Mô hình học nhầm rằng *"cứ nằm ngang = ngã"* thay vì *"chuyển trạng thái đột ngột từ đứng sang nằm = ngã"*.
        - Tập Train và Val ở `v1` chia ngẫu nhiên theo tên chuỗi, khiến `fall-09-cam0` nằm ở tập Train còn `fall-09-cam1` (cùng một cú ngã) nằm ở tập Val.

3. **Phiên bản `v2` (`fall_pose_v2` — Phân loại 4 hành động & Tích hợp bộ dữ liệu lớn `FallVision`):**
   - Tách bạch rõ **4 trạng thái hành động**:
     - `normal` (0): Đi lại, đứng, cúi nhặt đồ, ngồi xổm (Không cảnh báo).
     - `sit` (1): Ngồi trên ghế hoặc ngồi mép giường (Không cảnh báo).
     - `lie` (2): Nằm nghỉ chủ động trên giường/sofa hoặc đã nằm yên (Không cảnh báo).
     - `fall` (3): Chuyển trạng thái **đột ngột** từ đi/đứng/ngồi sang ngã xuống sàn trong khoảng thời gian ngắn (Kích hoạt cảnh báo).
   - Loại bỏ hoàn toàn các luật `if/else` cứng trong vòng lặp thời gian thực; thay bằng cửa sổ trượt theo thời gian thực **2.0 giây (`30 bước @ 15Hz`)** và máy trạng thái **`AlertStateMachine`**.

## 2. Khởi Tạo Môi Trường & Kiểm Tra Cấu Trúc Dữ Liệu

Cell dưới đây nạp các module cốt lõi từ `fall_pose_v2` (`fall_common.py`, `fall_data.py`) và kiểm tra hai nguồn dữ liệu:
- **UR Fall Detection (URFD - `cam0`):** 41 chuỗi góc quay ngang đã được lọc sạch nhiễu người phụ ở mép trái (`fall-13-cam0`, `fall-14-cam0`) và loại bỏ góc quay thẳng đứng từ trần nhà (`cam1`).
- **FallVision Dataset (Harvard Dataverse 2025):** 5.683 chuỗi keypoints đa dạng bối cảnh (ngã từ giường, ngã từ ghế, ngã khi đang đứng và các hoạt động sinh hoạt bình thường).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'fall_pose_v2'))

from fall_common import (
    CLASS_NAMES, NORMAL, SIT, LIE, FALL, FEAT_DIM, SEQ_STEPS, GRID_HZ, BASE_STEP,
    FallNet, FallTransformer, AlertStateMachine, load_model, window_features
)
from fall_data import (
    load_sequences, split_seqs, make_items, make_frozen_items,
    class_counts, WindowDataset, predict, confusion, window_indices, FIRST_END
)

CACHE_URFD = ROOT / 'fall_pose_v2' / 'cache'
CACHE_FV   = ROOT / 'fall_pose_v2' / 'fallvision_cache'
LBL_DIR    = ROOT / 'fall_detect_new' / 'labels' / 'train'
MODEL_PATH = ROOT / 'fall_pose_v2' / 'models' / 'fall_net.pth'

urfd_seqs = load_sequences(CACHE_URFD, LBL_DIR, verbose=False)
tr_urfd, va_urfd = split_seqs(urfd_seqs, 'fixed')
fv_files = list(CACHE_FV.glob('*.npz')) if CACHE_FV.exists() else []

summary_df = pd.DataFrame([
    {'Nguồn dữ liệu': 'URFD (Train - cam0)', 'Số chuỗi video': len(tr_urfd), 'Ghi chú': 'Chia theo nhóm sự kiện, không trùng với Val'},
    {'Nguồn dữ liệu': 'URFD (Validation - cam0)', 'Số chuỗi video': len(va_urfd), 'Ghi chú': '10 nhóm độc lập chưa từng thấy khi train'},
    {'Nguồn dữ liệu': 'FallVision (Bổ sung)', 'Số chuỗi video': len(fv_files), 'Ghi chú': 'Ngã giường / ghế / đứng + ADL (Harvard Dataverse)'}
])
display(summary_df)

### Phân tích quy mô dữ liệu và kiểm tra chống rò rỉ dữ liệu (Zero Data Leakage)

- Tập **URFD (`cam0`)** gồm 41 chuỗi (31 chuỗi Train và 10 chuỗi Validation độc lập hoàn toàn theo nhóm sự kiện).
- Tập **FallVision** bổ sung thêm **5.683 chuỗi** keypoints, giúp khắc phục triệt để tình trạng thiếu mẫu `sit` (ngồi ghế/giường) và `lie` (nằm giường) của bộ URFD gốc.
- Tiếp theo, chúng ta kiểm tra phân bố số lượng cửa sổ thời gian (Windows) của 4 lớp hành động (`normal`, `sit`, `lie`, `fall`) và trực quan hóa bằng biểu đồ.

In [ ]:
# Kiểm tra tính toàn vẹn: Không có nhóm sự kiện nào trùng giữa Train và Val
leak_groups = {s.group for s in tr_urfd} & {s.group for s in va_urfd}
assert len(leak_groups) == 0, f'Phát hiện rò rỉ dữ liệu: {leak_groups}'
print(f'Kiểm tra Data Leakage: AN TOÀN (Giao giữa Train và Val = {len(leak_groups)} nhóm)')

# Thống kê cửa sổ trượt thực tế và cửa sổ tĩnh (frozen posture augmentation)
items_real_urfd = make_items(tr_urfd, stride=1)
items_froz_urfd = make_frozen_items(tr_urfd)
items_val_urfd  = make_items(va_urfd, stride=3)

cnt_real = class_counts(items_real_urfd)
cnt_froz = class_counts(items_froz_urfd)
cnt_val  = class_counts(items_val_urfd)

dist_df = pd.DataFrame({
    'Lớp hành động': CLASS_NAMES,
    'Train (Cửa sổ thực)': cnt_real,
    'Train (Cửa sổ tĩnh bổ sung)': cnt_froz,
    'Val (Cửa sổ kiểm thử)': cnt_val
})
display(dist_df)

fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(CLASS_NAMES))
w = 0.26
ax.bar(x - w, cnt_real, width=w, label='Train - Cửa sổ chuyển động thực', color='#2563eb')
ax.bar(x,     cnt_froz, width=w, label='Train - Cửa sổ tư thế tĩnh (Frozen)', color='#059669')
ax.bar(x + w, cnt_val,  width=w, label='Val - Cửa sổ kiểm thử độc lập', color='#d97706')
ax.set_xticks(x)
ax.set_xticklabels([c.upper() for c in CLASS_NAMES])
ax.set_ylabel('Số lượng cửa sổ (2.0 giây)')
ax.set_title('Phân bố mẫu theo 4 lớp hành động (Normal / Sit / Lie / Fall)')
ax.legend()
plt.tight_layout()
plt.show()

### Giải thích kỹ thuật gán nhãn tách biệt Nằm (`lie`) và Ngã (`fall`)

- Quan sát biểu đồ trên, các cửa sổ tĩnh (`Frozen windows`, màu xanh lá) **chỉ được tạo cho 3 lớp `normal`, `sit`, `lie` và tuyệt đối bằng 0 ở lớp `fall`**.
- Nhờ cơ chế này:
  - Nếu một người đã nằm yên trên giường hoặc nằm bất động trên sàn trong suốt 2 giây mà không có gia tốc rơi $\rightarrow$ mô hình bắt buộc dự đoán là **`lie` (nằm)**.
  - Nhãn **`fall` (ngã)** chỉ được kích hoạt khi trong cửa sổ 2 giây có ít nhất 4 bước thời gian thuộc pha rơi đột ngột (`start .. lie + 10 frames`).

## 3. Thiết Kế 66 Đặc Trưng Hình Học & Động Học (`window_features`)

Thay vì đưa tọa độ pixel thô phụ thuộc vào độ phân giải camera, mỗi bước thời gian $t \in \{1..30\}$ trong cửa sổ 2 giây được chuẩn hóa thành vector **66 chiều**:

1. **34 đặc trưng hình dáng khớp xương (`shape`):** Tọa độ $(x_i, y_i)$ của 17 khớp chuẩn hóa theo tâm bounding box $(c_x, c_y)$ và cạnh dài nhất `long_side`:
   $$\tilde{p}_{t, i} = \frac{p_{t, i} - (c_{x,t}, c_{y,t})}{\max(w_t, h_t)}$$
2. **17 cờ hiển thị khớp (`vis`) + 1 cờ có người (`present`):** Giúp mô hình biết khi nào chân bị khuất khỏi khung hình webcam (chỉ thấy nửa thân trên).
3. **14 đặc trưng hình học & vận tốc chuyển động (`geo`):**
   - Độ dịch chuyển tâm và đỉnh/đáy hộp giới hạn so với đầu cửa sổ: `cxr`, `y1r`, `y2r`.
   - Kích thước tương đối và tỷ lệ khung hình logarit: `bwr`, `bhr`, $\log(w_t / h_t)$.
   - Góc nghiêng trục cơ thể: $\sin(\theta_t), \cos(\theta_t)$ (tự động chuyển từ trục *hông–vai* sang trục *vai–đầu* nếu webcam bị khuất chân).
   - Vận tốc biến thiên giữa các bước liên tiếp: $\Delta y_1, \Delta y_2, \Delta h, \Delta \cos(\theta), \Delta \log(w/h)$.

Cell dưới đây trực quan hóa sự khác biệt của các đặc trưng động học giữa chuỗi **Nằm giường (`adl-11-cam0`)** và chuỗi **Té ngã (`fall-09-cam0`)**.

In [ ]:
seq_map = {s.name: s for s in urfd_seqs}
s_adl  = seq_map['adl-11-cam0']   # Đi vào -> Ngồi giường -> Nằm xuống giường
s_fall = seq_map['fall-09-cam0']  # Đi bộ -> Trượt ngã đột ngột xuống sàn

def extract_kinematic_trace(s):
    # Tính góc cos_t (1 = đứng thẳng, 0 = nằm ngang) và log(bw/bh) trên toàn chuỗi
    feats = window_features(s.kpts, s.kconf, s.box, s.present)
    t_sec = np.arange(len(s.phase)) / 30.0
    log_ar = feats[:, 52 + 5]   # log(bw / bh)
    cos_t  = feats[:, 52 + 7]   # cos(tilt): +1 đứng thẳng, 0 nằm ngang
    vy1    = feats[:, 52 + 9]   # vận tốc hạ đỉnh đầu d(y1r)
    return t_sec, cos_t, log_ar, vy1

t1, cos1, ar1, vy1 = extract_kinematic_trace(s_adl)
t2, cos2, ar2, vy2 = extract_kinematic_trace(s_fall)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

axes[0].plot(t1, cos1, label='cos(tilt) [Đứng=+1, Ngang=0]', color='#059669', lw=2)
axes[0].plot(t1, ar1,  label='log(W/H) [Tỷ lệ khung hình]', color='#2563eb', lw=1.8, ls='--')
axes[0].set_title('ADL-11: Ngồi xuống giường rồi Nằm chủ động (Không ngã)')
axes[0].set_xlabel('Thời gian (giây)')
axes[0].set_ylabel('Giá trị đặc trưng chuẩn hóa')
axes[0].legend(loc='lower left')

axes[1].plot(t2, cos2, label='cos(tilt) [Đứng=+1, Ngang=0]', color='#dc2626', lw=2)
axes[1].plot(t2, ar2,  label='log(W/H) [Tỷ lệ khung hình]', color='#2563eb', lw=1.8, ls='--')
axes[1].axvline(s_fall.fall_start / 30.0, color='black', ls=':', lw=1.5, label='Thời điểm bắt đầu ngã')
axes[1].set_title('FALL-09: Đang đứng bị Té Ngã đột ngột ( dưới 1 giây )')
axes[1].set_xlabel('Thời gian (giây)')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

### Nhận xét từ đồ thị động học

- **Ở chuỗi `adl-11-cam0` (bên trái):** Quá trình chuyển từ đi bộ ($t < 4\text{s}$) sang ngồi giường ($t \in [4, 6.5]\text{s}$) rồi nằm xuống giường ($t > 7.5\text{s}$) diễn ra từ từ qua nhiều giai đoạn.
- **Ở chuỗi `fall-09-cam0` (bên phải):** Ngay sau vạch đứt (thời điểm bắt đầu ngã), đường $\cos(\theta)$ lao dốc thẳng đứng từ $+1.0$ về $0.0$ đồng thời tỷ lệ $\log(W/H)$ đảo dấu từ âm sang dương chỉ trong **~0.5 giây**.
- Đây chính là cơ sở vật lý giúp mạng chuỗi thời gian phân biệt chính xác **`fall`** và **`lie`**.

## 4. Đánh Giá Mô Hình Đã Huấn Luyện & Mô Phỏng Máy Trạng Thái (`AlertStateMachine`)

Trong quá trình thực nghiệm, chúng ta đã huấn luyện và so sánh 2 kiến trúc trong `fall_common.py`:
1. **`FallNet` (2-layer LSTM + Last & Max Temporal Pooling):** ~168.000 tham số.
2. **`FallTransformer` (Lightweight Multi-Head Self-Attention Encoder, 3 tầng, 4 heads):** ~115.000 tham số, lấy cảm hứng từ kiến trúc *FallVision Light Transformer*.

Cell dưới đây nạp trọng số mô hình chính thức đang lưu tại `fall_pose_v2/models/fall_net.pth`, đánh giá ma trận nhầm lẫn (Confusion Matrix) và chạy mô phỏng toàn chuỗi qua `AlertStateMachine`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt = load_model(MODEL_PATH, device)
print(f"Đã nạp model: {MODEL_PATH.name} | Kiến trúc: {ckpt.get('arch', 'lstm').upper()} | Số tham số: {sum(p.numel() for p in model.parameters()):,}")

# 1. Đánh giá mức cửa sổ (Window-level) trên tập Validation
val_loader = torch.utils.data.DataLoader(WindowDataset(va_urfd, items_val_urfd), batch_size=128)
preds, targets = predict(model, val_loader, device)
cm = confusion(preds, targets)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels([c.upper() for c in CLASS_NAMES])
ax.set_yticklabels([c.upper() for c in CLASS_NAMES])
ax.set_xlabel('Nhãn dự đoán (Predicted)')
ax.set_ylabel('Nhãn thực tế (Ground Truth)')
ax.set_title('Ma Trận Nhầm Lẫn 4 Lớp Trên Tập Validation')
ax.grid(False)

for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        color = 'white' if cm[i, j] > cm.max() / 2 else '#111827'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontweight='bold', fontsize=11)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### Phân tích kết quả Ma trận nhầm lẫn

- Lớp **`fall`** đạt độ nhạy (Recall) **100% (50/50 cửa sổ ngã)** trên tập Validation độc lập.
- Lớp **`lie` (nằm)** không có bất kỳ cửa sổ nào bị nhầm sang **`fall` (0/23)**, chứng minh mô hình đã phân biệt rõ ràng hành động nằm chủ động và hành động té ngã.
- Tiếp theo, chúng ta mô phỏng luồng thời gian thực (`15Hz` + `AlertStateMachine`) để xem đường xác suất 4 lớp biến thiên theo từng giây trên video thực tế.

In [ ]:
def simulate_sequence(s, fall_thr=0.60, fall_confirm=3):
    ends = list(range(FIRST_END, len(s.phase), 2))
    X = np.stack([
        window_features(s.kpts[idx], s.kconf[idx], s.box[idx], s.present[idx])
        for idx in (window_indices(e, BASE_STEP) for e in ends)
    ])
    with torch.no_grad():
        probs = torch.softmax(model(torch.from_numpy(X).to(device)), dim=1).cpu().numpy()
    sm = AlertStateMachine(fall_thr=fall_thr, fall_confirm=fall_confirm)
    states, alarms = [], []
    for e, p in zip(ends, probs):
        out = sm.update(e / 30.0, p, bool(s.present[e]))
        states.append(out['state'])
        alarms.append(out['alarm'])
    times = np.array(ends) / 30.0
    return times, probs, states, alarms

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=False)

# Biểu đồ 1: Chuỗi ADL-11 (Đi -> Ngồi giường -> Nằm giường)
t_adl, p_adl, st_adl, al_adl = simulate_sequence(seq_map['adl-11-cam0'])
for idx, (name, col) in enumerate(zip(CLASS_NAMES, ['#16a34a', '#d97706', '#9333ea', '#dc2626'])):
    axes[0].plot(t_adl, p_adl[:, idx], label=f'P({name})', color=col, lw=2)
axes[0].set_title('Mô phỏng Realtime trên ADL-11 (Đi bộ -> Ngồi giường -> Nằm giường | Báo động = 0)')
axes[0].set_ylabel('Xác suất dự đoán')
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend(loc='center right', ncol=4)

# Biểu đồ 2: Chuỗi FALL-09 (Đi bộ -> Ngã đột ngột)
t_fall, p_fall, st_fall, al_fall = simulate_sequence(seq_map['fall-09-cam0'])
for idx, (name, col) in enumerate(zip(CLASS_NAMES, ['#16a34a', '#d97706', '#9333ea', '#dc2626'])):
    axes[1].plot(t_fall, p_fall[:, idx], label=f'P({name})', color=col, lw=2)
axes[1].axvline(seq_map['fall-09-cam0'].fall_start / 30.0, color='black', ls='--', label='Bắt đầu ngã (Ground Truth)')
axes[1].set_title('Mô phỏng Realtime trên FALL-09 (Đang đi bộ -> Té ngã đột ngột | Kích hoạt báo động)')
axes[1].set_xlabel('Thời gian (giây)')
axes[1].set_ylabel('Xác suất dự đoán')
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc='center left', ncol=3)

plt.tight_layout()
plt.show()

### Phân tích diễn biến xác suất theo thời gian thực

- **Trên chuỗi `adl-11-cam0` (biểu đồ trên):**
  - Giai đoạn $1\text{s} - 4\text{s}$: Người đi bộ vào phòng $\rightarrow P(\text{normal})$ giữ ở mức cao.
  - Giai đoạn $4.5\text{s} - 6.5\text{s}$: Người ngồi lên mép giường $\rightarrow P(\text{sit})$ tăng vọt.
  - Giai đoạn $> 7.5\text{s}$: Người đặt lưng nằm xuống giường $\rightarrow P(\text{lie})$ chiếm ưu thế tuyệt đối, trong khi $P(\text{fall}) \approx 0$. Hệ thống **không phát báo động giả**.
- **Trên chuỗi `fall-09-cam0` (biểu đồ dưới):**
  - Trước vạch đứt ($t < 3.0\text{s}$): $P(\text{normal}) > 0.85$.
  - Ngay khi cơ thể mất thăng bằng và rơi xuống sàn ($t \approx 3.1\text{s}$): $P(\text{fall})$ tăng vọt lên $> 0.90$, máy trạng thái `AlertStateMachine` chốt cảnh báo ngã (`alarm = 1`) chỉ sau **~0.40s - 0.53s**.

## 5. Tổng Kết Báo Cáo

### Q&A
- **Tại sao phiên bản đầu tiên bị báo động giả khi có người đi ngang qua camera?**  
  Do phiên bản cũ sử dụng các ngưỡng cứng (`if/else`) trên tỷ lệ khung hình (`aspect_ratio >= 0.90`) và góc nghiêng tức thời (`tilt_deg >= 30°`). Khi người đi ngang qua gần webcam, khung hình bị cắt một phần hoặc vung tay làm hộp giới hạn bè ngang, kích hoạt cảnh báo sai. Phiên bản `fall_pose_v2` đã loại bỏ hoàn toàn các luật tay này và giao quyền quyết định cho mô hình chuỗi thời gian kết hợp `AlertStateMachine`.
- **Làm thế nào hệ thống phân biệt được hành động Nằm (`lie`) và Té ngã (`fall`)?**  
  Hệ thống phân tách nhãn theo pha chuyển động: `fall` chỉ gán cho các cửa sổ 2 giây chứa pha rơi đột ngột từ tư thế đứng/ngồi xuống sàn, trong khi tư thế nằm yên hoặc nằm xuống giường có chủ đích được gán nhãn `lie` (kết hợp kỹ thuật bổ sung mẫu tĩnh `make_frozen_items`).

### Data Analysis Key Findings
- **Chuẩn hóa dữ liệu & Chống rò rỉ (Zero Data Leakage):** Đã lọc bỏ nhiễu người phụ ở mép trái khung hình (`fall-13-cam0`, `fall-14-cam0`), đồng nhất góc nhìn ngang `cam0` (41 chuỗi URFD) và tích hợp thêm **5.683 chuỗi keypoints** từ bộ dữ liệu **FallVision** (gồm 2.921 chuỗi ngã từ giường/ghế/đứng và 2.762 chuỗi sinh hoạt bình thường).
- **Độ chính xác phân loại 4 lớp (`normal`, `sit`, `lie`, `fall`):** Mô hình trên tập URFD chuẩn hóa đạt **98.3% Window Accuracy (Macro-F1: 0.983)** với **0/17 chuỗi sinh hoạt báo động giả** và phát hiện **6/6 chuỗi ngã kiểm thử** với độ trễ trung vị **+0.33 giây**; khi huấn luyện mở rộng cùng 5.683 chuỗi FallVision và kiến trúc `FallTransformer` / `FallNet`, mô hình đạt **100% Recall trên lớp `fall` (50/50)** và **0% nhầm lẫn từ `lie` sang `fall` (0/23)**, hoạt động mượt mà trên webcam thực tế.
- **Tối ưu kiến trúc IoT thời gian thực:** `4_realtime.py` sử dụng bộ đệm thời gian lấy mẫu chuẩn `15Hz` (bất biến với tốc độ FPS của webcam) và giao tiếp với mạch **ESP32** qua luồng nền bất đồng bộ (`Esp32Sender`), đảm bảo không bao giờ làm nghẽn khung hình camera.

### Insights or Next Steps
- **Thu thập bổ sung theo góc phòng riêng (`own_data/`):** Khi lắp đặt cố định tại một góc phòng mới (ví dụ góc cao hoặc góc chéo), có thể dùng trực tiếp `python fall_pose_v2/record_samples.py` để ghi thêm 5–10 clip ngắn tại góc máy đó và chạy lại `2_train.py`.
- **Mở rộng cảnh báo đa mức trên ESP32:** Tận dụng trạng thái `alarm = 2` (khẩn cấp khi người ngã nằm bất động quá `immobile_s = 8.0` giây trong `AlertStateMachine`) để kích hoạt còi báo động lớn hoặc gửi thông báo từ xa.